# 174 — Privacidad diferencial y aprendizaje federado

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución 1 — Laplace a mano

a) Conteo: una persona cambia el resultado a lo sumo en 1 → **Δf = 1**.

b) `b = Δf/ε = 1/0.25 = 4`; error esperado **E|Lap(4)| = 4** (≈ 24 % sobre 17: mucho
ruido, ε pequeño = mucha privacidad = poca precisión).

c) 12 publicaciones × 0.25 = **ε total = 3.0** → factor `e³ ≈ 20`. Una
distinguibilidad de 20× ya es una garantía muy débil: componer consultas "inocentes"
erosiona la privacidad, que es exactamente lo que la contabilidad de presupuesto
obliga a mirar.


In [ ]:
import random
random.seed(174)
b = 1 / 0.25
muestras = [random.expovariate(1 / b) - random.expovariate(1 / b) for _ in range(100000)]
error_medio = sum(abs(x) for x in muestras) / len(muestras)
print("b =", b, "| error medio simulado =", round(error_medio, 2))
import math
print("e^3 =", round(math.exp(3), 1))
assert abs(error_medio - 4) < 0.15


## Solución 2 — FedAvg a mano

a) `n = 500`; `w = (100·0.8 + 300·1.4 + 100·0.6)/500 = (80 + 420 + 60)/500 = 560/500`
= **1.12**.

b) Media simple = (0.8 + 1.4 + 0.6)/3 = **0.933**. En la ponderada domina B (60 % de
los datos); en la simple los tres pesan igual.

c) Si B es atípico (no-IID), la ponderación por n_k arrastra el modelo global hacia
su distribución: "más datos" no significa "más representativo". Es el problema
central de FedAvg con clientes heterogéneos — motiva variantes como FedProx o el
muestreo estratificado de clientes.


In [ ]:
clientes = {"A": (100, 0.8), "B": (300, 1.4), "C": (100, 0.6)}
n = sum(nk for nk, _ in clientes.values())
w_pond = sum(nk * w for nk, w in clientes.values()) / n
w_simple = sum(w for _, w in clientes.values()) / len(clientes)
print(round(w_pond, 3), round(w_simple, 3))
assert abs(w_pond - 1.12) < 1e-9


## Solución 3 — ¿Dónde está la fuga?

a) **FALSO.** Los gradientes/actualizaciones filtran información: la inversión de
gradientes (Deep Leakage from Gradients, arXiv:1906.08935) reconstruye ejemplos.

b) **VERDADERO.** Es el teorema de post-procesamiento: ninguna función sin acceso a
los datos puede debilitar la garantía.

c) **FALSO.** k-anonimato es una propiedad sintáctica de una tabla concreta; cae ante
datos auxiliares (Netflix Prize). ε-DP acota la inferencia contra cualquier auxiliar.

d) **VERDADERO.** SecAgg oculta actualizaciones individuales al servidor (amenaza:
servidor curioso), pero la suma agregada sigue sin ruido: sin DP no hay cota formal
sobre lo que la suma revela.


In [ ]:
respuestas = {
    "a": ("F", "los gradientes filtran: inversión de gradientes"),
    "b": ("V", ""),
    "c": ("F", "k-anonimato cae con datos auxiliares; no es garantía de mecanismo"),
    "d": ("V", ""),
}
assert [v[0] for v in respuestas.values()] == ["F", "V", "F", "V"]
print("consistente")


## Solución 4 — Diseño con presupuesto (referencia)

Un reparto defendible: ε = 0.6 para los 12 conteos (0.05 cada uno, ruido b = 20 —
tolerable si los conteos son grandes) y ε = 0.4 para la media anual, que se publica
una sola vez y alimenta decisiones de planificación (merece menos ruido relativo).
Alternativa estructural: sustituir los 12 conteos mensuales por un único histograma
anual por mes — una consulta de histograma disjunto consume un solo ε (cada persona
cae en un solo mes), reduciendo el gasto sin criptografía. Lo esencial: el reparto se
decide ANTES de consultar y se documenta, porque el presupuesto no es renovable.
